# 07 - Can the model see a single suspicious transaction?

Notebook 06 found that most missed positives have one labelled ACH transaction in an account that otherwise looks ordinary. Account totals may hide such a transaction. Two attempts are tried here.

- **A. More account features:** money passing through the account within a day, and how far the largest transaction stands out.
- **B. A transaction-level model:** every transaction gets a score, and an account takes the highest (`max`) or the combined (`noisy-or`) score of its transactions. `blend` averages this with the final model in log-odds; the aggregation and the weight are chosen on the validation fold of each round.

**Rule, fixed before the results were seen.** A candidate is accepted only if its mean validation AUC-PR gain over the final model is larger than the standard deviation of that gain over the five rounds. Test scores are reported but do not decide. If nothing passes, the final model stays as it is.

**Limit on how far this can be trusted.** The ideas come from notebook 06, which looked at errors on all folds. So even an accepted candidate would score somewhat optimistically, and no untouched data set is available here. Three fold seeds show how much a result depends on one split; they do not remove this bias.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_key,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.features import ALL_FEATURES, WINDOW_END, exchange_rates
from skeptical_second_reader.model import fit_final
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.split import (
    N_FOLDS,
    SEED,
    cached_folds,
    fold_roles,
    split_groups,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
groups = split_groups(owners, members)

X = pd.read_parquet(DATA_INTERIM / "experiment_features.parquet")[ALL_FEATURES]
saved = pd.read_parquet(DATA_INTERIM / "final_scores.parquet")
assert X.index.equals(saved.index)
y = labels.loc[X.index]

# Every feature below is computed from this frame, which has no label column.
unlabelled = df.drop(columns="is_laundering")
window = unlabelled[unlabelled["timestamp"] < WINDOW_END]
rates = exchange_rates(window)
tx = pd.DataFrame(
    {
        "sender": account_key(window["from_bank"], window["from_account"]).to_numpy(),
        "receiver": account_key(window["to_bank"], window["to_account"]).to_numpy(),
        "timestamp": window["timestamp"].to_numpy(),
        "payment_format": window["payment_format"].to_numpy(),
        "paid_currency": window["payment_currency"].to_numpy(),
        "received_currency": window["receiving_currency"].to_numpy(),
        "usd": (
            window["amount_paid"] * window["payment_currency"].map(rates)
        ).to_numpy(),
        "usd_received": (
            window["amount_received"] * window["receiving_currency"].map(rates)
        ).to_numpy(),
        "same_bank": (window["from_bank"] == window["to_bank"]).to_numpy(),
    }
)
tx_label = df.loc[window.index, "is_laundering"].to_numpy()
is_self = (tx["sender"] == tx["receiver"]).to_numpy()
print(
    f"accounts {len(X):,}  positives {y.sum():,}  "
    f"window transactions {len(tx):,}  labelled {tx_label.sum():,}"
)

accounts 515,078  positives 6,347  window transactions 5,077,237  labelled 4,522
CPU times: total: 18.4 s
Wall time: 16.1 s


## 1. The inputs

For every send, the latest money received by the same account in the previous 24 hours is looked up. The send counts as pass-through if its amount is 80% to 105% of that money. Self-transactions are left out. The same lookup forwards in time gives what the receiver does next.

In [2]:
%%time
DAY = pd.Timedelta("24h")
PASS_LOW, PASS_HIGH = 0.8, 1.05
moves = tx.assign(row=np.arange(len(tx)))[~is_self]
ordered = moves.sort_values("timestamp", kind="stable")

incoming = (
    ordered[["receiver", "timestamp", "usd_received"]]
    .rename(columns={"receiver": "account", "usd_received": "in_usd"})
    .assign(in_time=lambda t: t["timestamp"])
)
before = pd.merge_asof(
    ordered[["row", "sender", "timestamp", "usd", "payment_format"]].rename(
        columns={"sender": "account"}
    ),
    incoming,
    on="timestamp",
    by="account",
    direction="backward",
    tolerance=DAY,
)
before["hold_hours"] = (
    before["timestamp"] - before["in_time"]
).dt.total_seconds() / 3600
before["ratio"] = before["usd"] / before["in_usd"]
before["is_pass"] = before["ratio"].between(PASS_LOW, PASS_HIGH)

outgoing = (
    ordered[["sender", "timestamp", "usd"]]
    .rename(columns={"sender": "account", "usd": "next_usd"})
    .assign(next_time=lambda t: t["timestamp"])
)
after = pd.merge_asof(
    ordered[["row", "receiver", "timestamp", "usd_received"]].rename(
        columns={"receiver": "account"}
    ),
    outgoing,
    on="timestamp",
    by="account",
    direction="forward",
    tolerance=DAY,
)
after["wait_hours"] = (
    after["next_time"] - after["timestamp"]
).dt.total_seconds() / 3600
after["next_ratio"] = after["next_usd"] / after["usd_received"]

sent = moves.groupby("sender")["usd"]
received = moves.groupby("receiver")["usd_received"]
ach_moves = moves[moves["payment_format"] == "ACH"]
pass_rows = before[before["is_pass"]]
new_columns = {
    "pass_through_tx": pass_rows.groupby("account").size(),
    "pass_through_ach_tx": pass_rows[pass_rows["payment_format"] == "ACH"]
    .groupby("account")
    .size(),
    "pass_through_share": pass_rows.groupby("account").size() / sent.size(),
    "log_pass_through_usd": np.log1p(pass_rows.groupby("account")["usd"].sum()),
    "median_hold_hours": before.groupby("account")["hold_hours"].median(),
    "log_max_over_median_sent": np.log1p(sent.max()) - np.log1p(sent.median()),
    "log_max_over_median_received": np.log1p(received.max())
    - np.log1p(received.median()),
    "largest_sent_share": sent.max() / (sent.sum() + 1),
    "largest_received_share": received.max() / (received.sum() + 1),
    "log_max_ach_sent_usd": np.log1p(ach_moves.groupby("sender")["usd"].max()),
    "log_max_ach_received_usd": np.log1p(
        ach_moves.groupby("receiver")["usd_received"].max()
    ),
}
NEW_FEATURES = list(new_columns)
new_table = pd.DataFrame(
    {name: values.reindex(X.index) for name, values in new_columns.items()}
)
zero_when_absent = [c for c in NEW_FEATURES if c != "median_hold_hours"]
new_table[zero_when_absent] = new_table[zero_when_absent].fillna(0)
X_extended = pd.concat([X, new_table], axis=1)
print("new account features:", len(NEW_FEATURES))
print(new_table.describe().T[["mean", "50%", "max"]].round(3).to_string())

new account features: 11
                               mean    50%       max
pass_through_tx               0.153  0.000  2352.000
pass_through_ach_tx           0.015  0.000    13.000
pass_through_share            0.006  0.000     1.000
log_pass_through_usd          0.378  0.000    20.239
median_hold_hours             9.364  8.733    24.000
log_max_over_median_sent      0.649  0.000    19.356
log_max_over_median_received  0.966  0.000    20.266
largest_sent_share            0.319  0.099     1.000
largest_received_share        0.220  0.066     1.000
log_max_ach_sent_usd          2.532  0.000    24.003
log_max_ach_received_usd      1.783  0.000    24.003
CPU times: total: 9.23 s
Wall time: 9.27 s


The pass-through count is checked against a plain loop on a random sample of accounts, written separately from the lookup above.

In [3]:
sample = pd.Series(X.index[X["out_tx"] > 0]).sample(300, random_state=0)
sends_of = moves[moves["sender"].isin(sample)].sort_values("timestamp", kind="stable")
ins_of = ordered[ordered["receiver"].isin(sample)]
mismatches = 0
for account in sample:
    sends_here = sends_of[sends_of["sender"] == account]
    ins_here = ins_of[ins_of["receiver"] == account]
    count = 0
    for when, usd in zip(sends_here["timestamp"], sends_here["usd"], strict=True):
        seen = ins_here[
            (ins_here["timestamp"] <= when) & (ins_here["timestamp"] >= when - DAY)
        ]
        if len(seen) and PASS_LOW <= usd / seen["usd_received"].iloc[-1] <= PASS_HIGH:
            count += 1
    mismatches += count != new_table.loc[account, "pass_through_tx"]
print("accounts checked:", len(sample), " mismatches:", mismatches)
assert mismatches == 0

accounts checked: 300  mismatches: 0


In [4]:
%%time


def code_of(values):
    return pd.factorize(values, sort=True)[0].astype("float32")


def per_row(frame, column):
    values = np.full(len(tx), np.nan, dtype="float32")
    values[frame["row"].to_numpy()] = frame[column].to_numpy(dtype="float32")
    return values


n_out = tx["sender"].map(tx["sender"].value_counts())
n_in = tx["receiver"].map(tx["receiver"].value_counts())
links = tx[["sender", "receiver"]].drop_duplicates()
out_degree = tx["sender"].map(links["sender"].value_counts())
in_degree = tx["receiver"].map(links["receiver"].value_counts())
pair_count = tx.groupby(["sender", "receiver"])["usd"].transform("size")
log_usd = np.log1p(tx["usd"])
log_received = np.log1p(tx["usd_received"])
conversion = is_self & (tx["paid_currency"] != tx["received_currency"]).to_numpy()
conversion_keys = tx.loc[
    conversion, ["sender", "timestamp", "received_currency"]
].drop_duplicates()
conversion_keys = conversion_keys.rename(
    columns={"received_currency": "paid_currency"}
).assign(after_conversion=1.0)
after_conversion = (
    tx[["sender", "timestamp", "paid_currency"]]
    .merge(conversion_keys, on=["sender", "timestamp", "paid_currency"], how="left")[
        "after_conversion"
    ]
    .fillna(0)
    .to_numpy()
)
sender_conversions = (
    tx["sender"].map(tx.loc[conversion, "sender"].value_counts()).fillna(0)
)

T = pd.DataFrame(
    {
        "log_usd": log_usd,
        "payment_format": code_of(tx["payment_format"]),
        "paid_currency": code_of(tx["paid_currency"]),
        "changes_currency": tx["paid_currency"] != tx["received_currency"],
        "is_self": is_self,
        "same_bank": tx["same_bank"],
        "hour": tx["timestamp"].dt.hour,
        "weekday": tx["timestamp"].dt.day_of_week,
        "sender_tx": n_out,
        "receiver_tx": n_in,
        "sender_out_degree": out_degree,
        "receiver_in_degree": in_degree,
        "pair_count": pair_count,
        "log_usd_over_sender_median": log_usd
        - log_usd.groupby(tx["sender"]).transform("median"),
        "log_usd_over_receiver_median": log_received
        - log_received.groupby(tx["receiver"]).transform("median"),
        "after_conversion": after_conversion,
        "sender_conversions": sender_conversions,
        "hours_since_money_in": per_row(before, "hold_hours"),
        "ratio_to_money_in": per_row(before, "ratio"),
        "hours_to_next_send": per_row(after, "wait_hours"),
        "next_send_ratio": per_row(after, "next_ratio"),
    }
).astype("float32")
sender_position = X.index.get_indexer(tx["sender"])
receiver_position = X.index.get_indexer(tx["receiver"])
assert (sender_position >= 0).all() and (receiver_position >= 0).all()
print("transaction features:", T.shape[1])
print(f"memory: {T.memory_usage().sum() / 1e9:.2f} GB")

transaction features: 21
memory: 0.43 GB
CPU times: total: 20.6 s
Wall time: 20.6 s


## 2. The rounds

The folds and roles are those of the final model. The transaction model trains only on transactions with both ends in training accounts and stops early on transactions with both ends in validation accounts. A transaction that touches a test account could reveal that account's label, so it is never trained on. It is scored afterwards. Validation-fold account scores use the same scored transactions, so validation and test are treated alike. The transaction model is not stacked into the account model: training accounts would get scores from a model that saw them, the mismatch found for degrees in notebook 05.

In [5]:
TX_PARAMS = {
    "n_estimators": 1000,
    "learning_rate": 0.1,
    "max_depth": 6,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "early_stopping_rounds": 50,
    "random_state": SEED,
}
WEIGHTS = [0.25, 0.5, 0.75]


def logit(values):
    clipped = np.clip(values, 1e-7, 1 - 1e-7)
    return np.log(clipped / (1 - clipped))


def account_scores(tx_scores):
    highest = np.zeros(len(X))
    log_miss = np.zeros(len(X))
    miss = np.log1p(-np.clip(tx_scores, 0, 1 - 1e-7))
    other_side = ~is_self
    np.maximum.at(highest, sender_position, tx_scores)
    np.maximum.at(highest, receiver_position[other_side], tx_scores[other_side])
    np.add.at(log_miss, sender_position, miss)
    np.add.at(log_miss, receiver_position[other_side], miss[other_side])
    return {"max": highest, "noisy-or": 1 - np.exp(log_miss)}


def fit_tx(train_rows, val_rows, target):
    model = XGBClassifier(**TX_PARAMS)
    model.fit(
        T[train_rows],
        target[train_rows],
        eval_set=[(T[val_rows], target[val_rows])],
        verbose=False,
    )
    return model


def run_rounds(fold_of, target=tx_label, with_accounts=True):
    rows, kept = [], {}
    y_array = y.to_numpy()
    for k in range(N_FOLDS):
        roles = fold_roles(fold_of, k).to_numpy()
        val, test = roles == "val", roles == "test"
        train = roles == "train"
        sender_role, receiver_role = roles[sender_position], roles[receiver_position]
        train_rows = (sender_role == "train") & (receiver_role == "train")
        val_rows = (sender_role == "val") & (receiver_role == "val")
        model = fit_tx(train_rows, val_rows, target)
        aggregated = account_scores(model.predict_proba(T)[:, 1])
        candidates = dict(aggregated)
        info = {
            "round": k,
            "train_tx": int(train_rows.sum()),
            "train_labelled": int(target[train_rows].sum()),
            "val_labelled": int(target[val_rows].sum()),
            "best_iteration": model.best_iteration,
        }
        if with_accounts:
            final = fit_final(X[train], y[train], X[val], y[val])
            extended = fit_final(X_extended[train], y[train], X_extended[val], y[val])
            final_scores = final(X)
            candidates["final"] = final_scores
            candidates["A: more account features"] = extended(X_extended)
            best_blend, best_value, best_name = None, -1, None
            for name, values in aggregated.items():
                for weight in WEIGHTS:
                    mixed = weight * logit(final_scores) + (1 - weight) * logit(values)
                    value = average_precision_score(y_array[val], mixed[val])
                    if value > best_value:
                        best_blend, best_value = mixed, value
                        best_name = f"{name} w={weight}"
            candidates["B: blend"] = best_blend
            info["blend"] = best_name
        for name, values in candidates.items():
            info[f"val {name}"] = average_precision_score(y_array[val], values[val])
            info[f"test {name}"] = average_precision_score(y_array[test], values[test])
            kept.setdefault(name, np.full(len(X), np.nan))[test] = values[test]
        rows.append(info)
    return pd.DataFrame(rows), {n: pd.Series(v, index=X.index) for n, v in kept.items()}

In [6]:
%%time
RENAME = {"max": "B: tx model, max", "noisy-or": "B: tx model, noisy-or"}


def decision_table(rounds):
    names = [c[4:] for c in rounds.columns if c.startswith("val ") and c != "val final"]
    rows = []
    for name in names:
        val_gain = rounds[f"val {name}"] - rounds["val final"]
        test_gain = rounds[f"test {name}"] - rounds["test final"]
        rows.append(
            {
                "candidate": RENAME.get(name, name),
                "val": rounds[f"val {name}"].mean(),
                "val_gain": val_gain.mean(),
                "val_gain_sd": val_gain.std(ddof=1),
                "accepted": val_gain.mean() > val_gain.std(ddof=1),
                "test": rounds[f"test {name}"].mean(),
                "test_gain": test_gain.mean(),
                "test_gain_sd": test_gain.std(ddof=1),
            }
        )
    return pd.DataFrame(rows).set_index("candidate").round(4)


fold_of = cached_folds(labels, groups, DATA_INTERIM).loc[X.index]
rounds, kept = run_rounds(fold_of)
assert rounds["test final"].round(4).tolist() == [
    0.3812,
    0.3451,
    0.3552,
    0.3601,
    0.3777,
]
print("final model equals notebook 04: True")
print(
    f"final: val {rounds['val final'].mean():.4f}, "
    f"test {rounds['test final'].mean():.4f}"
)
print(
    rounds[
        [
            "round",
            "train_tx",
            "train_labelled",
            "val_labelled",
            "best_iteration",
            "blend",
        ]
    ].to_string(index=False)
)
print()
print(decision_table(rounds).to_string())

final model equals notebook 04: True
final: val 0.3637, test 0.3639
 round  train_tx  train_labelled  val_labelled  best_iteration           blend
     0   1914421            2186           587             241 noisy-or w=0.25
     1   2042098            2384           557             141  noisy-or w=0.5
     2   2009457            2221           647             107 noisy-or w=0.25
     3   1996278            2199           553             137 noisy-or w=0.25
     4   1951343            2191           603             209  noisy-or w=0.5

                             val  val_gain  val_gain_sd  accepted    test  test_gain  test_gain_sd
candidate                                                                                         
B: tx model, max          0.4719    0.1082       0.0169      True  0.4710     0.1072        0.0178
B: tx model, noisy-or     0.4824    0.1187       0.0128      True  0.4806     0.1167        0.0147
A: more account features  0.3807    0.0170       0.0065      

## 3. Which groups change?

Each candidate flags as many accounts per round as the final model flagged, so the comparison does not depend on a threshold. The groups are those of notebook 06. `conversion rule` is exactly one conversion with an ACH send in the same minute. The last columns score only accounts outside that rule.

In [7]:
sender_all = account_key(df["from_bank"], df["from_account"])
receiver_all = account_key(df["to_bank"], df["to_account"])
in_window = df["timestamp"] < WINDOW_END
self_all = sender_all == receiver_all
bad = in_window & (df["is_laundering"] == 1)
ach = df["payment_format"] == "ACH"


def touches(mask):
    counted = pd.concat([sender_all[mask], receiver_all[mask & ~self_all]])
    return counted.value_counts().reindex(X.index, fill_value=0)


bad_count = touches(bad)
ach_bad_count = touches(bad & ach)
positive = y == 1
conversions_per_account = (
    tx.loc[conversion, "sender"].value_counts().reindex(X.index, fill_value=0)
)
ach_after = (
    (after_conversion == 1) & ~is_self & (tx["payment_format"] == "ACH").to_numpy()
)
rule = pd.Series(X.index.isin(tx.loc[ach_after, "sender"]), index=X.index) & (
    conversions_per_account == 1
)
print("conversion rule accounts:", int(rule.sum()), " positives:", int(y[rule].sum()))

report_groups = {
    "all positives": positive,
    "one ACH label, one currency": positive
    & (ach_bad_count > 0)
    & (bad_count == 1)
    & (X["n_currencies"] == 1),
    "labelled, none ACH": positive & (bad_count > 0) & (ach_bad_count == 0),
    "no label in the window": positive & (bad_count == 0),
    "2+ labels, some ACH": positive & (ach_bad_count > 0) & (bad_count >= 2),
    "conversion rule": positive & rule,
}
print({name: int(mask.sum()) for name, mask in report_groups.items()})

budget_of = saved.groupby("fold")["flag"].sum()
rows = []
for name, values in kept.items():
    for k in range(N_FOLDS):
        test = fold_of == k
        chosen = values[test].nlargest(int(budget_of[k])).index
        outside = test & ~rule
        row = {
            "candidate": RENAME.get(name, name),
            "precision": y[chosen].mean(),
            "outside rule auc_pr": average_precision_score(y[outside], values[outside]),
            "outside rule auc_roc": roc_auc_score(y[outside], values[outside]),
        }
        for group_name, mask in report_groups.items():
            row[group_name] = mask[chosen].sum() / (test & mask).sum()
        rows.append(row)
group_table = pd.DataFrame(rows)
print("\nmean over rounds: recall per group at the same budget")
print(group_table.groupby("candidate", sort=False).mean().T.round(4).to_string())
print("\nstandard deviation over rounds")
print(group_table.groupby("candidate", sort=False).std(ddof=1).T.round(4).to_string())

conversion rule accounts: 1187  positives: 1097
{'all positives': 6347, 'one ACH label, one currency': 3457, 'labelled, none ACH': 740, 'no label in the window': 485, '2+ labels, some ACH': 986, 'conversion rule': 1097}



mean over rounds: recall per group at the same budget
candidate                    B: tx model, max  B: tx model, noisy-or   final  A: more account features  B: blend
precision                              0.9219                 0.9265  0.7153                    0.7365    0.9312
outside rule auc_pr                    0.3289                 0.3386  0.1720                    0.1941    0.3427
outside rule auc_roc                   0.8990                 0.9101  0.8923                    0.8955    0.9137
all positives                          0.3386                 0.3405  0.2620                    0.2696    0.3420
one ACH label, one currency            0.2089                 0.2066  0.0771                    0.0841    0.1991
labelled, none ACH                     0.0000                 0.0178  0.0235                    0.0206    0.0190
no label in the window                 0.0000                 0.0000  0.0015                    0.0032    0.0000
2+ labels, some ACH                    0.

## 4. Controls

With shuffled transaction labels the transaction model should find nothing. Then the whole comparison runs on two other fold assignments, to show how much the result depends on one split.

In [8]:
%%time
shuffled = np.random.default_rng(SEED).permutation(tx_label)
control, control_scores = run_rounds(fold_of, target=shuffled, with_accounts=False)
print(
    "shuffled labels, test AUC-PR against the real labels (base rate",
    round(y.mean(), 4),
    ")",
)
print(control[["test max", "test noisy-or"]].agg(["mean", "std"]).round(4).to_string())

for fold_seed in [1, 2]:
    other = cached_folds(labels, groups, DATA_INTERIM, seed=fold_seed).loc[X.index]
    other_rounds, _ = run_rounds(other)
    print(
        f"\nfold seed {fold_seed}: "
        f"final val {other_rounds['val final'].mean():.4f}, "
        f"test {other_rounds['test final'].mean():.4f}"
    )
    print(decision_table(other_rounds).to_string())

shuffled labels, test AUC-PR against the real labels (base rate 0.0123 )
      test max  test noisy-or
mean    0.0193         0.0256
std     0.0018         0.0061



fold seed 1: final val 0.3609, test 0.3603
                             val  val_gain  val_gain_sd  accepted    test  test_gain  test_gain_sd
candidate                                                                                         
B: tx model, max          0.4679    0.1070       0.0109      True  0.4720     0.1117        0.0097
B: tx model, noisy-or     0.4780    0.1170       0.0100      True  0.4816     0.1213        0.0084
A: more account features  0.3794    0.0185       0.0054      True  0.3788     0.0185        0.0054
B: blend                  0.4839    0.1229       0.0077      True  0.4839     0.1236        0.0064



fold seed 2: final val 0.3647, test 0.3634
                             val  val_gain  val_gain_sd  accepted    test  test_gain  test_gain_sd
candidate                                                                                         
B: tx model, max          0.4682    0.1035       0.0191      True  0.4707     0.1073        0.0196
B: tx model, noisy-or     0.4783    0.1137       0.0162      True  0.4807     0.1174        0.0175
A: more account features  0.3816    0.0170       0.0074      True  0.3789     0.0155        0.0050
B: blend                  0.4844    0.1197       0.0149      True  0.4854     0.1220        0.0163
CPU times: total: 3h 9min 52s
Wall time: 13min 26s


## 5. What the transaction model looks at, and how well it ranks labelled transactions

This is one round's model, fitted again for its gain importance. Test transactions here have at least one end in a test account. It shows whether the signal the transaction model finds is the conversion rule again.

In [9]:
%%time
roles = fold_roles(fold_of, 0).to_numpy()
sender_role, receiver_role = roles[sender_position], roles[receiver_position]
train_rows = (sender_role == "train") & (receiver_role == "train")
val_rows = (sender_role == "val") & (receiver_role == "val")
test_rows = (sender_role == "test") | (receiver_role == "test")
model = fit_tx(train_rows, val_rows, tx_label)
importance = pd.Series(model.feature_importances_, index=T.columns).sort_values(
    ascending=False
)
print(importance.head(8).round(3).to_string())

tx_scores = model.predict_proba(T[test_rows])[:, 1]
target = tx_label[test_rows]
follows = after_conversion[test_rows] == 1
ach_rows = (tx["payment_format"] == "ACH").to_numpy()[test_rows]
print("\ntest transactions of round 0")
rows = {
    "all": np.ones(len(target), dtype=bool),
    "ACH, after a conversion": ach_rows & follows,
    "ACH, other": ach_rows & ~follows,
    "not ACH": ~ach_rows,
}
print(
    pd.DataFrame(
        {
            name: {
                "transactions": int(mask.sum()),
                "labelled": int(target[mask].sum()),
                "base_rate": target[mask].mean(),
                "auc_pr": average_precision_score(target[mask], tx_scores[mask]),
                "auc_roc": roc_auc_score(target[mask], tx_scores[mask]),
            }
            for name, mask in rows.items()
        }
    )
    .T.round(4)
    .to_string()
)

pair_count            0.346
after_conversion      0.251
payment_format        0.151
log_usd               0.072
hour                  0.054
sender_tx             0.029
sender_conversions    0.021
changes_currency      0.020



test transactions of round 0


                         transactions  labelled  base_rate  auc_pr  auc_roc
all                         1833310.0    1364.0     0.0007  0.3687   0.9632
ACH, after a conversion        3061.0     385.0     0.1258  0.9950   0.9994
ACH, other                   200795.0     559.0     0.0028  0.1158   0.9475
not ACH                     1629454.0     420.0     0.0003  0.0017   0.9114
CPU times: total: 8min 43s
Wall time: 41.6 s


## 6. Audit of the gain

The gain of the transaction model is too large to accept on the rule alone. Its largest feature is `pair_count`, the number of window transactions between the same sender and receiver, and it nearly separates labelled sends after a conversion. This section asks what that feature measures, what is left without it, whether a test account ever reaches training, and whether a simple count gives the same gain at account level. These checks were added after the first results were seen, so they do not count as a fixed rule.

In [10]:
for k in range(N_FOLDS):
    roles = fold_roles(fold_of, k).to_numpy()
    sender_role, receiver_role = roles[sender_position], roles[receiver_position]
    used = (sender_role == "train") & (receiver_role == "train")
    used |= (sender_role == "val") & (receiver_role == "val")
    in_fit = set(tx.loc[used, "sender"]) | set(tx.loc[used, "receiver"])
    assert not in_fit & set(X.index[fold_of == k])
print("no test account appears in a transaction used for fitting: True")

pair_bins = pd.cut(
    pair_count, [0, 1, 2, 5, 20, np.inf], labels=["1", "2", "3-5", "6-20", "21+"]
)
is_ach_tx = (tx["payment_format"] == "ACH").to_numpy()
views = {
    "ACH sends": is_ach_tx & ~is_self,
    "other sends": ~is_ach_tx & ~is_self,
    "ACH sends after a conversion": is_ach_tx & ~is_self & (after_conversion == 1),
}
for name, mask in views.items():
    part = pd.DataFrame({"bin": pair_bins[mask], "label": tx_label[mask]})
    print(f"\n{name}, by transactions between the same pair")
    print(
        part.groupby("bin", observed=True)["label"]
        .agg(["size", "sum", "mean"])
        .round(4)
        .to_string()
    )

hours = pd.DataFrame(
    {"hour": tx["timestamp"].dt.hour[is_ach_tx], "label": tx_label[is_ach_tx]}
)
by_hour = hours.groupby("hour")["label"].agg(["size", "mean"])
print(
    "\nACH by hour: lowest and highest labelled rate, and hours with the largest volume"
)
print(by_hour["mean"].agg(["min", "max"]).round(5).to_string())
print(by_hour.nlargest(3, "size").round(5).to_string())

no test account appears in a transaction used for fitting: True

ACH sends, by transactions between the same pair
        size   sum    mean
bin                       
1     111521  3176  0.0285
2      49741   360  0.0072
3-5    26693   108  0.0040
6-20  124115   152  0.0012
21+   216665    24  0.0001



other sends, by transactions between the same pair
         size  sum    mean
bin                       
1      110698   42  0.0004
2      222275   25  0.0001
3-5    213991   38  0.0002
6-20  2325021  391  0.0002
21+   1085698  198  0.0002

ACH sends after a conversion, by transactions between the same pair
      size   sum    mean
bin                     
1     1597  1498  0.9380
2      681   121  0.1777
3-5    451     0  0.0000
6-20  2227     0  0.0000
21+   3870     0  0.0000

ACH by hour: lowest and highest labelled rate, and hours with the largest volume
min    0.00312
max    0.01098
       size     mean
hour                
0     39049  0.00312
5     25009  0.00520
6     24914  0.00586


In [11]:
%%time
GROUPS = {
    "no pair_count": ["pair_count"],
    "no time of day": ["hour", "weekday"],
    "no conversion": [
        "after_conversion",
        "sender_conversions",
        "changes_currency",
        "is_self",
    ],
    "no pair_count, conversion": [
        "pair_count",
        "after_conversion",
        "sender_conversions",
        "changes_currency",
        "is_self",
    ],
    "no pair_count, conversion, time": [
        "pair_count",
        "after_conversion",
        "sender_conversions",
        "changes_currency",
        "is_self",
        "hour",
        "weekday",
    ],
}
y_array = y.to_numpy()
rows = []
variant_kept = {}
for name, removed in GROUPS.items():
    columns = [c for c in T.columns if c not in removed]
    for k in range(N_FOLDS):
        roles = fold_roles(fold_of, k).to_numpy()
        val, test = roles == "val", roles == "test"
        sender_role, receiver_role = roles[sender_position], roles[receiver_position]
        train_rows = (sender_role == "train") & (receiver_role == "train")
        val_rows = (sender_role == "val") & (receiver_role == "val")
        model = XGBClassifier(**TX_PARAMS)
        model.fit(
            T.loc[train_rows, columns],
            tx_label[train_rows],
            eval_set=[(T.loc[val_rows, columns], tx_label[val_rows])],
            verbose=False,
        )
        values = account_scores(model.predict_proba(T[columns])[:, 1])["noisy-or"]
        variant_kept.setdefault(name, np.full(len(X), np.nan))[test] = values[test]
        rows.append(
            {
                "variant": name,
                "round": k,
                "val_gain": average_precision_score(y_array[val], values[val])
                - rounds.loc[k, "val final"],
                "test": average_precision_score(y_array[test], values[test]),
                "test_gain": average_precision_score(y_array[test], values[test])
                - rounds.loc[k, "test final"],
                "outside_rule": average_precision_score(
                    y_array[test & ~rule.to_numpy()], values[test & ~rule.to_numpy()]
                ),
            }
        )
audit = pd.DataFrame(rows)
print(
    "transaction model (noisy-or) with features removed, gain against the final model"
)
print(
    audit.drop(columns="round")
    .groupby("variant", sort=False)
    .agg(["mean", "std"])
    .round(4)
    .to_string()
)

rows = []
for name, values in variant_kept.items():
    values = pd.Series(values, index=X.index)
    for k in range(N_FOLDS):
        test = fold_of == k
        chosen = values[test].nlargest(int(budget_of[k])).index
        row = {"variant": name, "precision": y[chosen].mean()}
        for group_name, mask in report_groups.items():
            row[group_name] = mask[chosen].sum() / (test & mask).sum()
        rows.append(row)
print("\nrecall per group at the budget of the final model")
print(pd.DataFrame(rows).groupby("variant", sort=False).mean().T.round(4).to_string())

transaction model (noisy-or) with features removed, gain against the final model
                                val_gain            test         test_gain         outside_rule        
                                    mean     std    mean     std      mean     std         mean     std
variant                                                                                                
no pair_count                     0.0792  0.0139  0.4373  0.0371    0.0735  0.0244       0.2908  0.0423
no time of day                    0.1102  0.0163  0.4717  0.0252    0.1078  0.0178       0.3295  0.0419
no conversion                     0.0409  0.0277  0.4009  0.0455    0.0370  0.0419       0.2876  0.0505
no pair_count, conversion        -0.0140  0.0133  0.3500  0.0192   -0.0138  0.0187       0.2532  0.0253
no pair_count, conversion, time  -0.0404  0.0167  0.3225  0.0289   -0.0413  0.0262       0.2313  0.0323



recall per group at the budget of the final model
variant                      no pair_count  no time of day  no conversion  no pair_count, conversion  no pair_count, conversion, time
precision                           0.8855          0.9177         0.7501                     0.6702                           0.6196
all positives                       0.3250          0.3367         0.2771                     0.2462                           0.2277
one ACH label, one currency         0.1908          0.1985         0.1874                     0.1686                           0.1507
labelled, none ACH                  0.0190          0.0190         0.0232                     0.0203                           0.0216
no label in the window              0.0000          0.0000         0.0032                     0.0047                           0.0032
2+ labels, some ACH                 0.8142          0.8248         0.7594                     0.7192                           0.6718
conversion 

The same signal at account level: for each account, how many ACH sends and receipts are its only transaction with that counterparty in the window. This is a plain count, it needs no transaction model, and the account model and SHAP stay as they are. It was thought of after seeing the results above, so its test score is the most exposed of all to the bias noted at the top.

In [12]:
%%time
single_pair = (pair_count == 1).to_numpy() & ~is_self
single_ach = single_pair & is_ach_tx
one_off = pd.DataFrame(
    {
        "ach_one_off_sent": tx.loc[single_ach, "sender"]
        .value_counts()
        .reindex(X.index, fill_value=0),
        "ach_one_off_received": tx.loc[single_ach, "receiver"]
        .value_counts()
        .reindex(X.index, fill_value=0),
        "other_one_off_sent": tx.loc[single_pair & ~is_ach_tx, "sender"]
        .value_counts()
        .reindex(X.index, fill_value=0),
        "other_one_off_received": tx.loc[single_pair & ~is_ach_tx, "receiver"]
        .value_counts()
        .reindex(X.index, fill_value=0),
    }
)
any_one_off = (one_off["ach_one_off_sent"] + one_off["ach_one_off_received"]) > 0
print(
    "accounts with a one-off ACH counterparty:",
    int(any_one_off.sum()),
    " positive rate:",
    round(y[any_one_off].mean(), 4),
)
print(
    "accounts without:",
    int((~any_one_off).sum()),
    " positive rate:",
    round(y[~any_one_off].mean(), 4),
)
print("share of positives with one:", round(any_one_off[y == 1].mean(), 4))

X_one_off = pd.concat([X, one_off], axis=1)
rows = []
one_off_scores = pd.Series(np.nan, index=X.index)
for k in range(N_FOLDS):
    roles = fold_roles(fold_of, k)
    train, val, test = roles == "train", roles == "val", roles == "test"
    score = fit_final(X_one_off[train], y[train], X_one_off[val], y[val])
    one_off_scores[test] = score(X_one_off[test])
    val_value = average_precision_score(y[val], score(X_one_off[val]))
    test_value = average_precision_score(y[test], one_off_scores[test])
    chosen = one_off_scores[test].nlargest(int(budget_of[k])).index
    outside = test & ~rule
    row = {
        "val_gain": val_value - rounds.loc[k, "val final"],
        "test": test_value,
        "test_gain": test_value - rounds.loc[k, "test final"],
        "outside_rule": average_precision_score(y[outside], one_off_scores[outside]),
        "precision": y[chosen].mean(),
    }
    for group_name, mask in report_groups.items():
        row[group_name] = mask[chosen].sum() / (test & mask).sum()
    rows.append(row)
print("\nfinal features plus the four one-off counts")
print(pd.DataFrame(rows).agg(["mean", "std"]).T.round(4).to_string())

pd.DataFrame(
    {
        "final": kept["final"],
        "tx_noisy_or": kept["noisy-or"],
        "blend": kept["B: blend"],
        "one_off": one_off_scores,
        "label": y,
        "fold": fold_of,
    }
).to_parquet(DATA_INTERIM / "tx_model_scores.parquet")

accounts with a one-off ACH counterparty: 120040  positive rate: 0.0413
accounts without: 395038  positive rate: 0.0035
share of positives with one: 0.7816



final features plus the four one-off counts
                               mean     std
val_gain                     0.0404  0.0163
test                         0.4014  0.0148
test_gain                    0.0375  0.0137
outside_rule                 0.2189  0.0405
precision                    0.7558  0.0944
all positives                0.2765  0.0261
one ACH label, one currency  0.0957  0.0345
labelled, none ACH           0.0249  0.0204
no label in the window       0.0015  0.0033
2+ labels, some ACH          0.7936  0.0460
conversion rule              0.9891  0.0061
CPU times: total: 21min 11s
Wall time: 1min 25s
